In [ ]:
# ===== PY 1.0 — Runtime, configuration, and shared route helpers =====
import asyncio, json, os, re, secrets, subprocess, time, unicodedata, uuid
from pathlib import Path
from urllib.parse import quote

import dialoghelper as dh
from fasthtml.common import *
from fasthtml.jupyter import *

port = 8000

def kill_port(port=port):
    subprocess.run(f'lsof -ti:{port} | xargs -r kill -9', shell=True)

if not globals().get('srv'):
    kill_port()
    app = FastHTML(session_cookie='solveit_social_session')
    rt = app.route
    srv = JupyUvi(app)
elif 'rt' not in globals():
    rt = app.route

SOCIAL_PORT = getattr(srv, 'port', port)
try:
    _domains = json.loads(os.environ.get('PUBLIC_DOMAINS', '{}'))
    if not isinstance(_domains, dict): _domains = {}
except (TypeError, json.JSONDecodeError):
    _domains = {}
public_domain = _domains.get(str(SOCIAL_PORT), globals().get('public_domain', ''))

def _drop_route(app, path, method=None):
    if app is not None:
        app.routes[:] = [route for route in app.routes if not (
            getattr(route, 'path', None) == path and
            (method is None or method in (getattr(route, 'methods', set()) or set())))]

def _endpoint(domain, path):
    base = str(domain or '').rstrip('/')
    if not base: return path
    return (base if base.startswith(('http://', 'https://')) else f'https://{base}') + path

def _cors(req):
    return {'Access-Control-Allow-Origin': req.headers.get('origin') or '*',
            'Vary': 'Origin', 'Cache-Control': 'no-store'}


In [ ]:
# ===== PY 2.0 — Read-only dialog-folder media service =====
IMAGES = {'.gif', '.jpeg', '.jpg', '.png', '.webp'}
VIDEOS = {'.m4v', '.mov', '.mp4', '.webm'}
MEDIA_ROUTE = '/social-media/files'

def install_social_files(rt, public_domain=None, path=MEDIA_ROUTE, app=None):
    app = app or getattr(rt, '__self__', None)
    _drop_route(app, path, 'GET')

    @rt(path)
    async def get(req, path: str = ''):
        root = Path(await dh.realpath(Path(dh.find_dname()).parent.as_posix())).resolve()
        data_root = Path(await dh.realpath('/')).resolve()
        relative, folder = Path(path or '.'), None
        try: folder = (root / relative).resolve()
        except OSError: pass
        if (relative.is_absolute() or folder is None or
            not folder.is_relative_to(root) or not folder.is_dir()):
            return JSONResponse({'error': 'Invalid folder.'}, status_code=400, headers=_cors(req))

        items = []
        for item in folder.iterdir():
            try: resolved = item.resolve()
            except OSError: continue
            if item.name.startswith('.') or not resolved.is_relative_to(root): continue
            suffix = item.suffix.lower()
            kind = ('folder' if item.is_dir() else 'gif' if suffix == '.gif' else
                    'image' if suffix in IMAGES else 'video' if suffix in VIDEOS else '')
            if not kind: continue
            entry = {'name': item.name, 'path': item.relative_to(root).as_posix(), 'kind': kind}
            if kind != 'folder':
                if not resolved.is_relative_to(data_root): continue
                entry['url'] = '/static/' + quote(resolved.relative_to(data_root).as_posix(), safe='/')
            items.append(entry)

        current = folder.relative_to(root).as_posix()
        current = '' if current == '.' else current
        parent = None if not current else Path(current).parent.as_posix()
        if parent == '.': parent = ''
        items.sort(key=lambda item: (item['kind'] != 'folder', item['name'].lower()))
        return JSONResponse({'path': current, 'parent': parent, 'items': items}, headers=_cors(req))

    iife(f'window.SOLVEIT_MEDIA_API_URL={json.dumps(_endpoint(public_domain, path))}')


In [ ]:
# ===== PY 3.0 — Shared live-publishing transport and media inspection =====
MB = 1024 * 1024
SOLVEIT_SOCIAL_PUBLISH_TOKEN = globals().get('SOLVEIT_SOCIAL_PUBLISH_TOKEN') or globals().get('SOCIAL_PUBLISH_TOKEN') or secrets.token_urlsafe(32)
SOCIAL_PUBLISH_TOKEN = SOLVEIT_SOCIAL_PUBLISH_TOKEN
X_PUBLISH_LOCK = globals().get('X_PUBLISH_LOCK') or globals().get('SOLVEIT_SOCIAL_PUBLISH_LOCK') or asyncio.Lock()
X_PUBLISH_ATTEMPTS = globals().get('X_PUBLISH_ATTEMPTS') or globals().get('SOLVEIT_SOCIAL_PUBLISH_ATTEMPTS') or {}
LINKEDIN_PUBLISH_LOCK = globals().get('LINKEDIN_PUBLISH_LOCK') or asyncio.Lock()
LINKEDIN_PUBLISH_ATTEMPTS = globals().get('LINKEDIN_PUBLISH_ATTEMPTS') or {}
SOLVEIT_SOCIAL_PUBLISH_LOCK, SOLVEIT_SOCIAL_PUBLISH_ATTEMPTS = X_PUBLISH_LOCK, X_PUBLISH_ATTEMPTS

X_MEDIA = {
    'image/jpeg': ('image', '.jpg', 5 * MB), 'image/png': ('image', '.png', 5 * MB),
    'image/webp': ('image', '.webp', 5 * MB), 'image/gif': ('gif', '.gif', 15 * MB),
    'video/mp4': ('video', '.mp4', 512 * MB), 'video/quicktime': ('video', '.mov', 512 * MB),
    'video/x-m4v': ('video', '.m4v', 512 * MB), 'video/webm': ('video', '.webm', 512 * MB),
}
LIVE_MEDIA = X_MEDIA  # backward-compatible X name
LINKEDIN_MEDIA = {
    'image/jpeg': ('image', '.jpg', 512 * MB), 'image/png': ('image', '.png', 512 * MB),
    'image/gif': ('gif', '.gif', 512 * MB), 'video/mp4': ('video', '.mp4', 500 * MB),
}
MAX_LIVE_MEDIA = 512 * MB

def _media_kind(head):
    if head.startswith(b'\xff\xd8\xff'): return 'image', 'image/jpeg'
    if head.startswith(b'\x89PNG\r\n\x1a\n'): return 'image', 'image/png'
    if head[:4] == b'RIFF' and head[8:12] == b'WEBP': return 'image', 'image/webp'
    if head.startswith((b'GIF87a', b'GIF89a')): return 'gif', 'image/gif'
    if head[4:8] == b'ftyp':
        brand = head[8:12]
        if brand == b'qt  ': return 'video', 'video/quicktime'
        if brand.startswith(b'M4V'): return 'video', 'video/x-m4v'
        return 'video', 'video/mp4'
    if head.startswith(b'\x1aE\xdf\xa3'): return 'video', 'video/webm'
    return '', ''

async def _read_live_media(upload, planned, platform='x'):
    """Inspect an UploadFile without loading large videos into memory."""
    file = getattr(upload, 'file', None)
    if file is None: raise ValueError('An uploaded media item could not be read.')
    def inspect():
        file.seek(0, 2); size = file.tell(); file.seek(0); head = file.read(32); file.seek(0)
        return size, head
    size, head = await asyncio.to_thread(inspect)
    if not size: raise ValueError('A selected media item is empty.')
    kind, detected = _media_kind(head)
    supplied = str(getattr(upload, 'content_type', '') or '').lower().split(';')[0]
    formats = LINKEDIN_MEDIA if platform == 'linkedin' else X_MEDIA
    if not kind or detected not in formats:
        allowed = 'a JPEG, PNG, GIF, or MP4 file' if platform == 'linkedin' else 'a JPEG, PNG, WebP, GIF, MP4, MOV, M4V, or WebM file'
        raise ValueError(f'Use {allowed}.')
    if kind != planned['kind']:
        raise ValueError(f"Media marked as {planned['kind']} does not match its contents.")
    if supplied in formats and formats[supplied][0] != kind:
        raise ValueError('A selected media type does not match its contents.')
    if platform == 'linkedin' and kind == 'video':
        if size < 75 * 1024: raise ValueError('LinkedIn video must be at least 75 KB.')
    mime = supplied if supplied in formats else detected
    limit = formats[mime][2]
    if size > limit: raise ValueError(f'The selected {kind} is larger than {limit // MB} MB.')
    if platform == 'linkedin' and kind in ('image', 'gif'):
        def inspect_image():
            try: from PIL import Image
            except ImportError as error:
                raise ValueError('Install Pillow before publishing LinkedIn images.') from error
            try:
                with Image.open(file) as image:
                    pixels = image.width * image.height
                    frames = getattr(image, 'n_frames', 1)
            except Exception as error:
                raise ValueError('A selected LinkedIn image could not be inspected.') from error
            finally: file.seek(0)
            if pixels >= 36_152_320:
                raise ValueError('LinkedIn images must contain fewer than 36,152,320 pixels.')
            if kind == 'gif' and frames > 250:
                raise ValueError('LinkedIn GIFs can contain at most 250 frames.')
        await asyncio.to_thread(inspect_image)
    original = re.split(r'[/\\]', str(getattr(upload, 'filename', '') or kind))[-1]
    stem = re.sub(r'[^A-Za-z0-9._-]+', '_', re.sub(r'\.[^.]*$', '', original))[:80] or kind
    return {'file': file, 'kind': kind, 'mime': mime, 'size': size,
            'name': stem + formats[mime][1], 'altText': str(planned.get('altText') or '').strip()}

def _failure(message, status=500, **details):
    result = {'live': True, 'success': False, 'complete': False, 'safeToRetry': True,
              'resultUnknown': False, 'error': message, 'posts': [], 'sideEffects': [],
              'retryPerformed': False}
    return result | details, status

async def _publish_request(req, platform, planner, publisher, lock, attempts):
    headers, uploads = _cors(req), {}
    def reject(message, status, **details):
        return JSONResponse({'live': True, 'success': False, 'error': message} | details,
                            status_code=status, headers=headers)
    try:
        if req.headers.get('content-type', '').lower().startswith('multipart/form-data'):
            try: form = await req.form(max_files=400, max_fields=2, max_part_size=MAX_LIVE_MEDIA + 1024)
            except TypeError: form = await req.form(max_files=400, max_fields=2)
            body = json.loads(str(form.get('payload') or ''))
            for key, value in form.multi_items():
                if key.startswith('media_'):
                    if key in uploads: raise ValueError('Duplicate media upload field.')
                    uploads[key] = value
        else: body = json.loads((await req.body()).decode())
    except Exception:
        return reject('A JSON object is required.', 400)
    if not isinstance(body, dict): return reject('A JSON object is required.', 400)
    supplied = str(body.pop('publishToken', ''))
    if not supplied or not secrets.compare_digest(supplied, SOCIAL_PUBLISH_TOKEN):
        return reject('Publishing is not authorized.', 403)
    request_id = str(body.pop('requestId', ''))
    try: uuid.UUID(request_id)
    except (ValueError, AttributeError):
        return reject('A valid request ID is required.', 400)
    attempted = 'xAttempted' if platform == 'x' else 'linkedinAttempted'
    try:
        plan = planner(body, len(uploads))
        expected = {f'media_{i}_{j}': (i, item) for i, step in enumerate(plan)
                    for j, item in enumerate(step['media'])}
        if set(uploads) != set(expected): raise ValueError('The uploaded media fields do not match the posting plan.')
        media = [[] for _ in plan]
        for key, (post_index, planned) in expected.items():
            media[post_index].append(await _read_live_media(uploads[key], planned, platform))
    except (ValueError, TypeError) as error:
        return reject(str(error), 422, **{attempted: False}, safeToRetry=True,
                      resultUnknown=False, posts=[], sideEffects=[], retryPerformed=False)
    if lock.locked():
        return reject('Another publish request is already running.', 409, **{attempted: False},
                      posts=[], sideEffects=[], retryPerformed=False)
    async with lock:
        if request_id in attempts:
            return reject('This publish request was already attempted.', 409, **{attempted: False},
                          posts=[], sideEffects=[], retryPerformed=False)
        attempts[request_id] = True
        while len(attempts) > 100: attempts.pop(next(iter(attempts)))
        result, status = await asyncio.to_thread(publisher, plan, media)
    return JSONResponse(result, status_code=status, headers=headers)

def _install_publish_route(rt, path, platform, planner, publisher, lock, attempts, app=None):
    app = app or getattr(rt, '__self__', None)
    _drop_route(app, path)
    @rt(path)
    async def post(req):
        return await _publish_request(req, platform, planner, publisher, lock, attempts)


In [ ]:
# ===== PY 4.0 — X adapter =====
X_SECRET_NAMES = ('X_API_KEY', 'X_API_SECRET', 'X_ACCESS_TOKEN', 'X_ACCESS_TOKEN_SECRET')
PUBLISH_ROUTE = '/social-media/publish'

def _live_plan(body, media_count=0):
    """Revalidate a confirmed live thread before any X request."""
    if not isinstance(body, dict): raise ValueError('A JSON object is required.')
    thread, validation, confirmation = body.get('thread'), body.get('validation'), body.get('confirmation')
    if not isinstance(thread, dict) or not isinstance(validation, dict):
        raise ValueError('thread and validation objects are required.')
    posts, metrics = thread.get('posts'), validation.get('posts')
    if (thread.get('schemaVersion') != 1 or thread.get('platform') != 'x' or
        thread.get('charLimit') != 280 or not isinstance(posts, list) or not 1 <= len(posts) <= 100 or
        not isinstance(metrics, list) or len(metrics) != len(posts)):
        raise ValueError('Unsupported, malformed, or stale thread payload.')
    if not isinstance(confirmation, dict) or confirmation.get('action') != 'publish_to_x' or confirmation.get('postCount') != len(posts):
        raise ValueError('Explicit publishing confirmation is required.')
    expected_type = 'single' if len(posts) == 1 else 'thread'
    if thread.get('postType') != expected_type: raise ValueError(f'postType must be {expected_type}.')

    plan, ids, total = [], set(), 0
    for index, (post, metric) in enumerate(zip(posts, metrics), 1):
        if (not isinstance(post, dict) or not isinstance(post.get('text'), str) or
            not isinstance(post.get('media'), list) or not isinstance(metric, dict)):
            raise ValueError(f'Post {index} is malformed.')
        weighted, remaining = metric.get('weightedLength'), metric.get('remaining')
        if (not isinstance(weighted, int) or isinstance(weighted, bool) or weighted < 0 or
            not isinstance(remaining, int) or remaining != 280 - weighted):
            raise ValueError(f'Post {index} validation is stale.')
        if weighted > 280: raise ValueError(f'Post {index} is over its character limit.')
        client_id, text, items = str(post.get('clientId') or ''), post['text'], post['media']
        if not client_id or client_id in ids: raise ValueError(f'Post {index} needs a unique clientId.')
        if not text.strip() and not items: raise ValueError(f'Post {index} is empty.')
        if len(items) > 4: raise ValueError(f'Post {index} has more than four media items.')
        ids.add(client_id); refs, media = set(), []
        for media_index, item in enumerate(items, 1):
            if not isinstance(item, dict) or item.get('kind') not in ('image', 'gif', 'video'):
                raise ValueError(f'Post {index}, media {media_index} is invalid.')
            ref = str(item.get('ref') or '')
            if not ref or item.get('missing') or ref in refs:
                raise ValueError(f'Post {index} has unresolved or duplicate media.')
            refs.add(ref); media.append({'kind': item['kind'], 'altText': str(item.get('altText') or '')})
        total += len(media); plan.append({'clientId': client_id, 'text': text, 'media': media})
    if media_count != total: raise ValueError('The uploaded media does not match the posting plan.')
    return plan

def _x_auth():
    try: import tweepy
    except ImportError as error: raise RuntimeError('Install Tweepy 4.14 or newer before live publishing.') from error
    missing = [name for name in X_SECRET_NAMES if not os.environ.get(name)]
    if missing: raise RuntimeError('Missing X credentials: ' + ', '.join(missing))
    return tweepy, [os.environ[name] for name in X_SECRET_NAMES]

def _x_client():
    tweepy, values = _x_auth()
    return tweepy.Client(consumer_key=values[0], consumer_secret=values[1],
        access_token=values[2], access_token_secret=values[3], wait_on_rate_limit=False)

def _x_media_api():
    tweepy, values = _x_auth()
    return tweepy.API(tweepy.OAuth1UserHandler(*values))

def _x_post_id(response):
    data = response.get('data', response) if isinstance(response, dict) else getattr(response, 'data', None)
    value = data.get('id') if isinstance(data, dict) else getattr(data, 'id', None)
    if not value: raise RuntimeError('X returned no post ID.')
    return str(value)

def _x_error(error):
    response = getattr(error, 'response', None)
    status, message = getattr(response, 'status_code', None), ''
    try:
        data = response.json()
        message = data.get('detail') or data.get('title') or next((x.get('message', '') for x in data.get('errors', [])), '')
    except Exception: pass
    return status, re.sub(r'\s+', ' ', message or str(error) or 'X rejected the request.').strip()[:500]

def _publish_plan(plan, client=None, media_api=None, media=None):
    """Upload all media, then create a non-retried reply chain in order."""
    media = media or [[] for _ in plan]
    if len(media) != len(plan) or any(len(files) != len(step['media']) for files, step in zip(media, plan)):
        return _failure('Prepared media does not match the posting plan.', 422, xAttempted=False)
    if client is None:
        try: client = _x_client()
        except Exception as error:
            _, message = _x_error(error)
            return _failure(message, xAttempted=False)
    effects, media_ids = [], [[] for _ in plan]
    if any(media):
        if media_api is None:
            try: media_api = _x_media_api()
            except Exception as error:
                _, message = _x_error(error)
                return _failure(message, xAttempted=False, postAttempted=False)
        for post_index, files in enumerate(media):
            for media_index, item in enumerate(files):
                try:
                    chunked, category = item['kind'] != 'image', f"tweet_{item['kind']}"
                    uploaded = media_api.media_upload(filename=item['name'], file=item['file'], chunked=chunked,
                        media_category=category, **({'wait_for_async_finalize': True} if chunked else {}))
                    processing = getattr(uploaded, 'processing_info', None) or {}
                    problem = processing.get('error')
                    if problem: raise RuntimeError(problem.get('message') if isinstance(problem, dict) else str(problem))
                    if processing.get('state') not in (None, 'succeeded'): raise RuntimeError('X did not finish processing the media.')
                    media_id = str(getattr(uploaded, 'media_id_string', None) or getattr(uploaded, 'media_id', ''))
                    if not media_id: raise RuntimeError('X returned no media ID.')
                    media_ids[post_index].append(media_id)
                    effects.append({'action': 'upload_media', 'id': media_id, 'postStep': post_index + 1,
                                    'mediaIndex': media_index, 'kind': item['kind']})
                    if item['altText'] and item['kind'] != 'video':
                        media_api.create_media_metadata(media_id, item['altText'][:1000])
                except Exception as error:
                    status, message = _x_error(error)
                    return _failure(message, 502, xAttempted=True, postAttempted=False,
                        mediaUploadAttempted=True, failedStep=post_index + 1, failedMedia=media_index,
                        xStatus=status, sideEffects=effects)

    created, previous = [], None
    for index, step in enumerate(plan):
        try:
            kwargs = {'user_auth': True}
            if step['text'].strip(): kwargs['text'] = step['text']
            if media_ids[index]: kwargs['media_ids'] = media_ids[index]
            if previous: kwargs['in_reply_to_tweet_id'] = previous
            post_id = _x_post_id(client.create_tweet(**kwargs))
            created.append({'step': index + 1, 'clientId': step['clientId'], 'id': post_id,
                'url': f'https://x.com/i/web/status/{post_id}', 'replyToId': previous,
                'mediaIds': media_ids[index]})
            effects.append({'action': 'create_post', 'id': post_id, 'step': index + 1})
            previous = post_id
        except Exception as error:
            status, message = _x_error(error); unknown = status is None
            return _failure(message, 502, partial=bool(created), xAttempted=True, postAttempted=True,
                mediaUploadAttempted=bool(any(media)), safeToRetry=not created and not unknown,
                resultUnknown=unknown, failedStep=index + 1, publishedCount=len(created),
                xStatus=status, posts=created, sideEffects=effects)

    return {'live': True, 'success': True, 'complete': True, 'xAttempted': True, 'resultUnknown': False,
        'postAttempted': True, 'mediaUploadAttempted': bool(any(media)), 'safeToRetry': False,
        'postCount': len(created), 'publishedCount': len(created), 'rootPostId': created[0]['id'],
        'lastPostId': created[-1]['id'], 'posts': created, 'sideEffects': effects, 'retryPerformed': False}, 200

def install_social_publish(rt, public_domain=None, path=PUBLISH_ROUTE, app=None):
    def publish(plan, media): return _publish_plan(plan, None, None, media)
    _install_publish_route(rt, path, 'x', _live_plan, publish,
                           X_PUBLISH_LOCK, X_PUBLISH_ATTEMPTS, app)
    iife(f'delete window.SOLVEIT_SOCIAL_DRY_RUN_URL;window.SOLVEIT_SOCIAL_PUBLISH_URL={json.dumps(_endpoint(public_domain, path))};'
         f'window.SOLVEIT_SOCIAL_PUBLISH_TOKEN={json.dumps(SOCIAL_PUBLISH_TOKEN)}')


In [ ]:
# ===== PY 5.0 — LinkedIn adapter =====
LINKEDIN_ROUTE = '/social-media/linkedin/publish'
LINKEDIN_VERSION = os.environ.get('LINKEDIN_VERSION', '202607')
LINKEDIN_SECRET_NAMES = ('LINKEDIN_ACCESS_TOKEN',)

def _linkedin_plan(body, media_count=0):
    if not isinstance(body, dict): raise ValueError('A JSON object is required.')
    draft, validation, confirmation = body.get('thread'), body.get('validation'), body.get('confirmation')
    if not isinstance(draft, dict) or not isinstance(validation, dict):
        raise ValueError('thread and validation objects are required.')
    posts, metrics = draft.get('posts'), validation.get('posts')
    if (draft.get('schemaVersion') != 1 or draft.get('platform') != 'linkedin' or
        draft.get('postType') != 'single' or draft.get('charLimit') != 3000 or
        not isinstance(posts, list) or len(posts) != 1 or
        not isinstance(metrics, list) or len(metrics) != 1):
        raise ValueError('Unsupported, malformed, or stale LinkedIn payload.')
    if not isinstance(confirmation, dict) or confirmation.get('action') != 'publish_to_linkedin' or confirmation.get('postCount') != 1:
        raise ValueError('Explicit LinkedIn publishing confirmation is required.')
    post, metric = posts[0], metrics[0]
    if (not isinstance(post, dict) or not isinstance(post.get('text'), str) or
        not isinstance(post.get('media'), list) or not isinstance(metric, dict)):
        raise ValueError('The LinkedIn post is malformed.')
    text, items = unicodedata.normalize('NFC', post['text']), post['media']
    length = len(text)
    weighted, remaining = metric.get('weightedLength'), metric.get('remaining')
    if (not isinstance(weighted, int) or isinstance(weighted, bool) or weighted != length or
        not isinstance(remaining, int) or isinstance(remaining, bool) or remaining != 3000 - length):
        raise ValueError('LinkedIn character validation is stale.')
    if length > 3000: raise ValueError('The LinkedIn post is over its character limit.')
    if not text.strip() and not items: raise ValueError('The LinkedIn post is empty.')
    if len(items) > 4: raise ValueError('This composer supports up to four images or one video for LinkedIn.')
    client_id = str(post.get('clientId') or '')
    if not client_id: raise ValueError('The LinkedIn post needs a clientId.')
    refs, media = set(), []
    for index, item in enumerate(items, 1):
        if not isinstance(item, dict) or item.get('kind') not in ('image', 'gif', 'video'):
            raise ValueError(f'LinkedIn media {index} is invalid.')
        ref = str(item.get('ref') or '')
        if not ref or item.get('missing') or ref in refs:
            raise ValueError('The LinkedIn post has unresolved or duplicate media.')
        alt = str(item.get('altText') or '')
        if len(alt) > 4086: raise ValueError(f'LinkedIn media {index} alt text is longer than 4,086 characters.')
        refs.add(ref); media.append({'kind': item['kind'], 'altText': alt})
    videos = sum(item['kind'] == 'video' for item in media)
    if videos > 1 or videos and len(media) > 1:
        raise ValueError('LinkedIn posts can contain images or one video, not a mixture.')
    if media_count != len(media): raise ValueError('The uploaded media does not match the posting plan.')
    return [{'clientId': client_id, 'text': text, 'media': media}]

def _linkedin_config():
    try: import httpx
    except ImportError as error: raise RuntimeError('Install httpx before LinkedIn publishing.') from error
    missing = [name for name in LINKEDIN_SECRET_NAMES if not os.environ.get(name)]
    if missing: raise RuntimeError('Missing LinkedIn credentials: ' + ', '.join(missing))
    author = os.environ.get('LINKEDIN_AUTHOR_URN', '').strip()
    if author and not re.fullmatch(r'urn:li:(?:person|organization):[^:\s]+', author):
        raise RuntimeError('LINKEDIN_AUTHOR_URN must be a person or organization URN.')
    if not re.fullmatch(r'\d{6}', LINKEDIN_VERSION):
        raise RuntimeError('LINKEDIN_VERSION must use YYYYMM format.')
    mode = os.environ.get('LINKEDIN_API_MODE', 'auto').strip().lower()
    if mode not in ('auto', 'rest', 'legacy'):
        raise RuntimeError('LINKEDIN_API_MODE must be auto, rest, or legacy.')
    return httpx, os.environ['LINKEDIN_ACCESS_TOKEN'], author, mode

def _linkedin_author(client, token, configured=''):
    """Use an explicit author, or lazily discover the authenticated member."""
    if configured: return configured
    response = client.get('https://api.linkedin.com/v2/userinfo',
                          headers={'Authorization': f'Bearer {token}', 'Accept': 'application/json'})
    if response.status_code == 401:
        raise RuntimeError('The LinkedIn access token is invalid or expired.')
    if response.status_code == 403:
        raise RuntimeError('Automatic LinkedIn author discovery requires openid and profile. Add LINKEDIN_AUTHOR_URN or issue a token with those scopes.')
    value = _linkedin_json(response, {200})
    member_id = str(value.get('sub') or '').strip() if isinstance(value, dict) else ''
    if not re.fullmatch(r'[^:\s]+', member_id):
        raise RuntimeError('LinkedIn userinfo returned no valid member ID. Add LINKEDIN_AUTHOR_URN or issue a token with openid and profile.')
    return f'urn:li:person:{member_id}'

def _linkedin_headers(token):
    return {'Authorization': f'Bearer {token}', 'Linkedin-Version': LINKEDIN_VERSION,
            'X-Restli-Protocol-Version': '2.0.0', 'Content-Type': 'application/json'}

def _linkedin_error(error):
    response = getattr(error, 'response', None)
    status, message = getattr(response, 'status_code', None), ''
    try:
        data = response.json()
        if isinstance(data, dict):
            message = data.get('message') or data.get('code') or data.get('error_description') or ''
        elif isinstance(data, str): message = data
    except Exception:
        try: message = response.text
        except Exception: pass
    return status, re.sub(r'\s+', ' ', message or str(error) or 'LinkedIn rejected the request.').strip()[:500]

def _linkedin_json(response, expected):
    if response.status_code not in expected:
        response.raise_for_status()
    try: return response.json()
    except Exception: return {}

def _file_chunks(file, start=0, length=None, chunk=1024 * 1024):
    file.seek(start); remaining = length
    while remaining is None or remaining > 0:
        data = file.read(chunk if remaining is None else min(chunk, remaining))
        if not data: break
        if remaining is not None: remaining -= len(data)
        yield data
    if remaining not in (None, 0): raise RuntimeError('A media upload ended before the expected byte range.')

def _linkedin_upload_image(client, token, author, item):
    initialized = _linkedin_json(client.post('https://api.linkedin.com/rest/images?action=initializeUpload',
        headers=_linkedin_headers(token), json={'initializeUploadRequest': {'owner': author}}), {200})
    value = initialized.get('value') or {}
    url, urn = value.get('uploadUrl'), value.get('image')
    if not url or not urn: raise RuntimeError('LinkedIn returned an invalid image upload instruction.')
    response = client.put(url, headers={'Authorization': f'Bearer {token}',
        'Content-Type': 'application/octet-stream', 'Content-Length': str(item['size'])},
        content=_file_chunks(item['file']))
    if response.status_code not in (200, 201): response.raise_for_status()
    return str(urn)

def _linkedin_upload_video(client, token, author, item):
    initialized = _linkedin_json(client.post('https://api.linkedin.com/rest/videos?action=initializeUpload',
        headers=_linkedin_headers(token), json={'initializeUploadRequest': {'owner': author,
        'fileSizeBytes': item['size'], 'uploadCaptions': False, 'uploadThumbnail': False}}), {200})
    value = initialized.get('value') or {}
    urn, upload_token = value.get('video'), value.get('uploadToken', '')
    instructions = value.get('uploadInstructions') or []
    if not urn or not instructions: raise RuntimeError('LinkedIn returned invalid video upload instructions.')
    part_ids, expected = [], 0
    for instruction in instructions:
        first, last, url = instruction.get('firstByte'), instruction.get('lastByte'), instruction.get('uploadUrl')
        if not isinstance(first, int) or not isinstance(last, int) or first != expected or last < first or not url:
            raise RuntimeError('LinkedIn returned an invalid video byte range.')
        length = last - first + 1
        response = client.put(url, headers={'Content-Type': 'application/octet-stream',
            'Content-Length': str(length)}, content=_file_chunks(item['file'], first, length))
        if response.status_code not in (200, 201): response.raise_for_status()
        etag = str(response.headers.get('etag') or '').strip().strip('"')
        if not etag: raise RuntimeError('LinkedIn returned no ETag for a video part.')
        part_ids.append(etag); expected = last + 1
    if expected != item['size']: raise RuntimeError('LinkedIn video upload instructions did not cover the whole file.')
    _linkedin_json(client.post('https://api.linkedin.com/rest/videos?action=finalizeUpload',
        headers=_linkedin_headers(token), json={'finalizeUploadRequest': {'video': urn,
        'uploadToken': upload_token, 'uploadedPartIds': part_ids}}), {200})
    return str(urn)

def _linkedin_wait_media(client, token, urn, kind, attempts=10, delay=1.5):
    """Confirm readiness when the token has read access; write-only tokens skip safely."""
    resource = 'videos' if kind == 'video' else 'images'
    url = f"https://api.linkedin.com/rest/{resource}/{quote(urn, safe='')}"
    for attempt in range(attempts):
        response = client.get(url, headers=_linkedin_headers(token))
        if response.status_code == 403: return None
        if response.status_code == 404:
            if attempt + 1 < attempts:
                time.sleep(delay); continue
            return None
        value = _linkedin_json(response, {200})
        status = str(value.get('status') or '').upper()
        if status == 'AVAILABLE': return True
        if status == 'PROCESSING_FAILED':
            raise RuntimeError(f'LinkedIn failed to process the {kind}.')
        if status not in ('WAITING_UPLOAD', 'PROCESSING', ''): return None
        if attempt + 1 < attempts: time.sleep(delay)
    return None

def _linkedin_commentary(text):
    """Render arbitrary user input as safe LinkedIn little text while retaining hashtags."""
    text, escaped = str(text), []
    hashtags = {index for match in re.finditer(r'(?<!\w)#[^\W#]+', text, re.UNICODE)
                for index in range(*match.span())}
    reserved = set('|{}@[]()<>\\#*_~')
    for index, char in enumerate(text):
        escaped.append(char if index in hashtags or char not in reserved else '\\' + char)
    return ''.join(escaped)

def _linkedin_success(step, post_id, urns, effects, verified=True, api_mode='rest'):
    url = f'https://www.linkedin.com/feed/update/{post_id}/'
    created = [{'step': 1, 'clientId': step['clientId'], 'id': post_id,
                'url': url, 'mediaIds': urns}]
    effects.append({'action': 'create_post', 'id': post_id, 'step': 1})
    return {'live': True, 'success': True, 'complete': verified, 'linkedinAttempted': True,
        'resultUnknown': False, 'postAttempted': True, 'mediaUploadAttempted': bool(urns),
        'processingUnverified': not verified, 'apiMode': api_mode,
        'safeToRetry': False, 'postCount': 1, 'publishedCount': 1, 'rootPostId': post_id,
        'lastPostId': post_id, 'posts': created, 'sideEffects': effects, 'retryPerformed': False}, 200

def _linkedin_legacy_headers(token):
    return {'Authorization': f'Bearer {token}', 'X-Restli-Protocol-Version': '2.0.0',
            'Content-Type': 'application/json'}

def _linkedin_legacy_upload_image(client, token, author, item):
    request = {'registerUploadRequest': {'recipes': ['urn:li:digitalmediaRecipe:feedshare-image'],
        'owner': author, 'serviceRelationships': [{'relationshipType': 'OWNER',
        'identifier': 'urn:li:userGeneratedContent'}]}}
    initialized = _linkedin_json(client.post('https://api.linkedin.com/v2/assets?action=registerUpload',
        headers=_linkedin_legacy_headers(token), json=request), {200})
    value = initialized.get('value') or {}
    mechanism = (value.get('uploadMechanism') or {}).get(
        'com.linkedin.digitalmedia.uploading.MediaUploadHttpRequest') or {}
    url, urn = mechanism.get('uploadUrl'), value.get('asset')
    if not url or not urn: raise RuntimeError('LinkedIn returned an invalid legacy image upload instruction.')
    response = client.put(url, headers={'Authorization': f'Bearer {token}',
        'Content-Type': item['mime'], 'Content-Length': str(item['size'])},
        content=_file_chunks(item['file']))
    if response.status_code not in (200, 201): response.raise_for_status()
    return str(urn)

def _linkedin_publish_legacy(plan, media, client, token, author):
    if any(item['kind'] == 'video' for item in media[0]):
        return _failure('LinkedIn legacy mode supports images only. Use LINKEDIN_API_MODE=rest for video.',
                        422, linkedinAttempted=False, apiMode='legacy')
    effects, urns = [], []
    try: author = _linkedin_author(client, token, author)
    except Exception as error:
        status, message = _linkedin_error(error)
        return _failure(message, 502, linkedinAttempted=True, identityLookupAttempted=True,
                        postAttempted=False, mediaUploadAttempted=False,
                        linkedinStatus=status, apiMode='legacy')
    for index, item in enumerate(media[0]):
        try:
            urn = _linkedin_legacy_upload_image(client, token, author, item)
            urns.append(urn); effects.append({'action': 'upload_media', 'id': urn,
                'postStep': 1, 'mediaIndex': index, 'kind': item['kind']})
        except Exception as error:
            status, message = _linkedin_error(error)
            return _failure(message, 502, linkedinAttempted=True, postAttempted=False,
                mediaUploadAttempted=True, failedMedia=index, linkedinStatus=status,
                sideEffects=effects, apiMode='legacy')
    step = plan[0]
    share = {'shareCommentary': {'text': step['text']},
             'shareMediaCategory': 'IMAGE' if urns else 'NONE'}
    if urns:
        share['media'] = [{'status': 'READY', 'media': urn,
            'title': {'text': item['name'][:200]},
            **({'description': {'text': item['altText']}} if item['altText'] else {})}
            for urn, item in zip(urns, media[0])]
    body = {'author': author, 'lifecycleState': 'PUBLISHED',
        'specificContent': {'com.linkedin.ugc.ShareContent': share},
        'visibility': {'com.linkedin.ugc.MemberNetworkVisibility': 'PUBLIC'}}
    try:
        response = client.post('https://api.linkedin.com/v2/ugcPosts',
                               headers=_linkedin_legacy_headers(token), json=body)
        created_data = _linkedin_json(response, {201})
        post_id = str(response.headers.get('x-restli-id') or
                      (created_data.get('id') if isinstance(created_data, dict) else '') or '')
        if not post_id:
            return _failure('LinkedIn accepted the post but returned no post ID. Check LinkedIn before trying again.',
                502, linkedinAttempted=True, postAttempted=True,
                mediaUploadAttempted=bool(media[0]), safeToRetry=False, resultUnknown=True,
                linkedinStatus=201, sideEffects=effects, apiMode='legacy')
    except Exception as error:
        status, message = _linkedin_error(error)
        unknown = status is None or status == 408 or (status is not None and status >= 500)
        return _failure(message, 502, linkedinAttempted=True, postAttempted=True,
            mediaUploadAttempted=bool(media[0]), safeToRetry=not unknown, resultUnknown=unknown,
            linkedinStatus=status, sideEffects=effects, apiMode='legacy')
    return _linkedin_success(step, post_id, urns, effects, verified=not urns, api_mode='legacy')

def _linkedin_publish_rest(plan, media, client, token, author):
    effects, urns, readiness = [], [], []
    try: author = _linkedin_author(client, token, author)
    except Exception as error:
        status, message = _linkedin_error(error)
        return _failure(message, 502, linkedinAttempted=True, identityLookupAttempted=True,
                        postAttempted=False, mediaUploadAttempted=False, linkedinStatus=status)
    for index, item in enumerate(media[0]):
        try:
            urn = (_linkedin_upload_video if item['kind'] == 'video' else _linkedin_upload_image)(
                client, token, author, item)
            urns.append(urn)
            effect = {'action': 'upload_media', 'id': urn, 'postStep': 1,
                      'mediaIndex': index, 'kind': item['kind'], 'readinessVerified': False}
            effects.append(effect)
            ready = _linkedin_wait_media(client, token, urn, item['kind'])
            readiness.append(ready); effect['readinessVerified'] = ready is True
        except Exception as error:
            status, message = _linkedin_error(error)
            return _failure(message, 502, linkedinAttempted=True, postAttempted=False,
                mediaUploadAttempted=True, failedMedia=index, linkedinStatus=status, sideEffects=effects)

    step = plan[0]
    body = {'author': author, 'commentary': _linkedin_commentary(step['text']),
        'visibility': 'PUBLIC', 'distribution': {'feedDistribution': 'MAIN_FEED',
        'targetEntities': [], 'thirdPartyDistributionChannels': []},
        'lifecycleState': 'PUBLISHED', 'isReshareDisabledByAuthor': False}
    if urns:
        if len(urns) == 1:
            content = {'id': urns[0]}
            if media[0][0]['kind'] == 'video': content['title'] = media[0][0]['name'][:200]
            elif media[0][0]['altText']: content['altText'] = media[0][0]['altText']
            body['content'] = {'media': content}
        else:
            body['content'] = {'multiImage': {'images': [
                {'id': urn, **({'altText': item['altText']} if item['altText'] else {})}
                for urn, item in zip(urns, media[0])]}}
    try:
        response = client.post('https://api.linkedin.com/rest/posts',
                               headers=_linkedin_headers(token), json=body)
        created_data = _linkedin_json(response, {201})
        post_id = str(response.headers.get('x-restli-id') or
                      (created_data.get('id') if isinstance(created_data, dict) else '') or '')
        if not post_id:
            return _failure('LinkedIn accepted the post but returned no post ID. Check LinkedIn before trying again.',
                502, linkedinAttempted=True, postAttempted=True,
                mediaUploadAttempted=bool(media[0]), safeToRetry=False, resultUnknown=True,
                linkedinStatus=201, sideEffects=effects)
    except Exception as error:
        status, message = _linkedin_error(error)
        unknown = status is None or status == 408 or (status is not None and status >= 500)
        return _failure(message, 502, linkedinAttempted=True, postAttempted=True,
            mediaUploadAttempted=bool(media[0]), safeToRetry=not unknown, resultUnknown=unknown,
            linkedinStatus=status, sideEffects=effects)
    verified = not readiness or all(value is True for value in readiness)
    return _linkedin_success(step, post_id, urns, effects, verified, 'rest')

def _linkedin_publish(plan, media, client=None):
    if len(plan) != 1 or len(media) != 1 or len(media[0]) != len(plan[0]['media']):
        return _failure('Prepared media does not match the LinkedIn plan.', 422, linkedinAttempted=False)
    try:
        httpx, token, author, mode = _linkedin_config()
    except Exception as error:
        _, message = _linkedin_error(error)
        return _failure(message, linkedinAttempted=False)
    def publish(active):
        if mode == 'legacy': return _linkedin_publish_legacy(plan, media, active, token, author)
        result, status = _linkedin_publish_rest(plan, media, active, token, author)
        result.setdefault('apiMode', 'rest')
        can_fallback = (mode == 'auto' and not media[0] and not result.get('success') and
            result.get('linkedinStatus') == 403 and not result.get('sideEffects') and
            not result.get('resultUnknown') and not any(item['kind'] == 'video' for item in media[0]))
        if can_fallback:
            result, status = _linkedin_publish_legacy(plan, media, active, token, author)
            result['fallbackUsed'] = True
        else: result['fallbackUsed'] = False
        return result, status
    if client is not None: return publish(client)
    try:
        with httpx.Client(timeout=httpx.Timeout(180, connect=20)) as owned:
            return publish(owned)
    except Exception as error:
        _, message = _linkedin_error(error)
        return _failure(message, linkedinAttempted=False)

def install_linkedin_publish(rt, public_domain=None, path=LINKEDIN_ROUTE, app=None):
    _install_publish_route(rt, path, 'linkedin', _linkedin_plan, _linkedin_publish,
                           LINKEDIN_PUBLISH_LOCK, LINKEDIN_PUBLISH_ATTEMPTS, app)
    iife(f'window.SOLVEIT_LINKEDIN_PUBLISH_URL={json.dumps(_endpoint(public_domain, path))};'
         f'window.SOLVEIT_LINKEDIN_PUBLISH_TOKEN={json.dumps(SOCIAL_PUBLISH_TOKEN)}')


In [ ]:
# ===== PY 6.0 — Idempotent route installation =====
# CRAFT runs every cell before the child dialog starts. These calls therefore
# install all routes and configure both browser publishing adapters automatically.
install_social_files(rt, public_domain, app=app)
install_social_publish(rt, public_domain, app=app)
install_linkedin_publish(rt, public_domain, app=app)
